# Lấy data và xuất csv (Bản full)

**Lấy data từ Google PLay**

In [4]:
# Cài đặt thư viện lấy dữ liệu Google Play
!pip install google-play-scraper

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
from google_play_scraper import reviews, Sort
import pandas as pd
import numpy as np

In [7]:
# Lấy ID của app cần lấy dữ liệu
app_id = 'com.mservice.momotransfer' # ID của MoMo

print(f"Đang bắt đầu lấy dữ liệu từ Google Play ({app_id})...")

# Bắt đầu lấy dữ liệu
result, continuation_token = reviews(
    app_id,
    lang='vi',             # Ngôn ngữ: Tiếng Việt
    country='vn',          # Quốc gia: Việt Nam
    sort=Sort.NEWEST,      # Lấy các bình luận theo thứ tự mới nhất
    count=30000,           # Số lượng
    filter_score_with=1    # Lấy các bình luận 1 sao để tìm ra các vấn đề người dùng hay phàn nàn
)

print("Đã lấy xong")

# Gán dữ liệu thu thập được vào Dataframe
df = pd.DataFrame(result)

Đang bắt đầu lấy dữ liệu từ Google Play (com.mservice.momotransfer)...
Đã lấy xong


In [8]:
df.head(5)

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,337309fa-0f6c-4a8a-9938-e8e7530ea62f,Người dùng Google,https://play-lh.googleusercontent.com/EGemoI2N...,kh cả đăng nhập dc,1,0,5.10.0,2026-06-27 03:48:26,None,NaT,5.10.0
1,6ed0b2b1-2814-44d8-a0d3-5c24e1f3fba4,Người dùng Google,https://play-lh.googleusercontent.com/EGemoI2N...,"tôi cảm thấy thất vọng về cách xử lý của momo,...",1,0,5.10.0,2026-06-27 02:22:50,None,NaT,5.10.0
2,c587f548-9e90-434a-bd58-49996fe75cb8,Người dùng Google,https://play-lh.googleusercontent.com/EGemoI2N...,"ví trả sau phế vật kinh tởm, tt 35k r nó bắt t...",1,0,5.10.0,2026-06-27 02:16:20,None,NaT,5.10.0
3,3ccf2bf9-f7ca-492a-bac6-1244688c56c1,Người dùng Google,https://play-lh.googleusercontent.com/EGemoI2N...,khốn kiếp cướp tiền,1,0,5.10.0,2026-06-27 02:08:47,None,NaT,5.10.0
4,b704bc71-4314-460c-9811-80faafb089c5,Người dùng Google,https://play-lh.googleusercontent.com/EGemoI2N...,trả tiền tao,1,0,5.10.0,2026-06-27 01:43:07,Cảm ơn bạn đã để lại đánh giá. MoMo rất tiếc k...,2026-06-27 02:00:02,5.10.0


In [9]:
df.tail(5)

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
29995,91486f70-35ac-408c-abf3-fe3cc22cb5cb,Người dùng Google,https://play-lh.googleusercontent.com/EGemoI2N...,Ko vào app được,1,2,4.0.11,2022-12-07 00:16:39,None,NaT,4.0.11
29996,4da743d2-08d2-4f7e-bfcf-1b0a70e4ff97,Người dùng Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Có việc thanh toán vào không được, cứ vào báo ...",1,0,2.1.34,2022-12-06 23:33:01,None,NaT,2.1.34
29997,ca2e5573-9da2-47e9-ad3d-dc519a906713,Người dùng Google,https://play-lh.googleusercontent.com/EGemoI2N...,Mắc gì phải đồng bộ danh bạ đt mới cho heo bạn...,1,0,4.0.10,2022-12-06 21:28:40,None,NaT,4.0.10
29998,05f4cab6-c043-4b88-b708-ac9f4fda2da0,Người dùng Google,https://play-lh.googleusercontent.com/EGemoI2N...,ok,1,3,4.0.11,2022-12-06 20:08:16,None,NaT,4.0.11
29999,14c22b55-612b-4964-9045-27a9968ac301,Người dùng Google,https://play-lh.googleusercontent.com/EGemoI2N...,Nạp ví hoàn cl,1,0,4.0.10,2022-12-06 18:51:33,None,NaT,4.0.10


In [10]:
# Lọc cột: Chỉ giữ lại nội dung, thời gian và lượng like
df = df[['content', 'at', 'thumbsUpCount']]

# Đổi lại tên cột cho dễ nhìn
df.columns = ['Comment', 'Time', 'Likes']

# Xử lý ngày tháng: Chuyển cột Time sang định dạng datetime để dễ xử lý sau này
df['Time'] = pd.to_datetime(df['Time'])

# Thêm cột độ dài comment để tiện lọc rác
df['Length'] = df['Comment'].apply(len)

print(f"Tổng số dòng lấy được: {len(df)}")
print(f"Dữ liệu từ ngày: {df['Time'].min()} -> đến ngày: {df['Time'].max()}")

Tổng số dòng lấy được: 30000
Dữ liệu từ ngày: 2022-12-06 18:51:33 -> đến ngày: 2026-06-27 03:48:26


In [11]:
# Kiểm tra xem có bao nhiêu comment có like (quan trọng cho dashboard)
liked_comments = df[df['Likes'] > 0]
print(f"Số lượng comment có người bấm like: {len(liked_comments)} ({len(liked_comments)/len(df)*100:.1f}%)")

Số lượng comment có người bấm like: 10561 (35.2%)


**Làm sạch và tiền xử lý (chưa tách từ)**

In [12]:
# Cài đặt thư viện xử lý emoji
!pip install -q emoji

# Nhập các thư viện tiền xử lý
import re
import emoji
import unicodedata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 14.5 MB/s eta 0:00:00


In [13]:
# Hàm làm sạch chuyên sâu
def clean_text_for_api(text):
    if pd.isna(text):
        return ""

    # Chuyển về dạng chuỗi
    text = str(text)

    # Chuẩn hóa Unicode (tránh lỗi font tiếng Việt)
    text = unicodedata.normalize('NFC', text)

    # Xóa Emoji (Nguyên nhân chính gây lỗi API JSON)
    text = emoji.replace_emoji(text, replace='')

    # Giữ lại tiếng Việt, số và dấu câu cơ bản
    # Regex này xóa các ký tự lạ, icon đặc biệt, chỉ giữ lại chữ, số và dấu câu
    text = re.sub(r'[^\w\s\.,\?!\:\(\)\"\-\%]', ' ', text)

    # Xóa khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# --- THỰC HIỆN LÀM SẠCH ---

print(f"Số lượng dòng gốc: {len(df)}")

# Áp dụng hàm làm sạch
print("Đang quét sạch emoji và ký tự rác...")
df['Comment'] = df['Comment'].apply(clean_text_for_api)

# Quan trọng: Xóa các dòng bị rỗng sau khi làm sạch
# (Ví dụ: comment chỉ toàn icon, xóa xong thành rỗng -> Bỏ)
df = df[df['Comment'].str.len() > 2]

print(f"Số lượng dòng sau khi làm sạch: {len(df)}")

Số lượng dòng gốc: 30000
Đang quét sạch emoji và ký tự rác...
Số lượng dòng sau khi làm sạch: 29571


**Xuất file**

In [14]:
import os

# Tạo thư mục nếu chưa có
base = '/content/drive/MyDrive/Customer Feedback Classifier'
os.makedirs(f'{base}/data/raw', exist_ok=True)
os.makedirs(f'{base}/data/processed', exist_ok=True)

# Xuất file
df.to_csv(f'{base}/data/raw/momo_full.csv', index=False, encoding='utf-8-sig')
print("Đã xuất file: 'momo_full.csv'")

Đã xuất file: 'momo_full.csv'


# Cắt dữ liệu từ file full

**Lấy 01/01/2025 làm mốc để cắt**

In [15]:
# Đường dẫn gốc trên Drive
base = '/content/drive/MyDrive/Customer Feedback Classifier'

# Đọc file full từ thư mục raw
df = pd.read_csv(f'{base}/data/raw/momo_full.csv')

# Chuyển cột Time sang định dạng datetime
df['Time'] = pd.to_datetime(df['Time'])

# Chọn mốc thời gian cắt: 01/01/2025
split_date = pd.to_datetime('2025-01-01')
end_date = pd.to_datetime('2025-12-06')

# Tập Train: Trước 2025
df_train = df[df['Time'] < split_date].copy()

# Tập Dashboard (Predict): Từ 01/01/2025 đến 06/12/2025
df_dashboard = df[(df['Time'] >= split_date) & (df['Time'] <= end_date)].copy()

print(f"Đã cắt xong")
print(f"- Tập Train (Train + Test): {len(df_train)} dòng (Dữ liệu cũ -> 31/12/2024)")
print(f"- Tập Dashboard (Để dự báo): {len(df_dashboard)} dòng (01/01/2025 -> 06/12/2025)")

Đã cắt xong
- Tập Train (Train + Test): 22309 dòng (Dữ liệu cũ -> 31/12/2024)
- Tập Dashboard (Để dự báo): 5000 dòng (01/01/2025 -> 06/12/2025)


In [16]:
import os
os.makedirs(f'{base}/data/processed', exist_ok=True)
os.makedirs(f'{base}/data/raw', exist_ok=True)
print("Đã tạo thư mục xong")

Đã tạo thư mục xong


In [17]:
df_train.to_csv(f'{base}/data/processed/momo_train.csv', index=False, encoding='utf-8-sig')
df_dashboard.to_csv(f'{base}/data/processed/momo_pred.csv', index=False, encoding='utf-8-sig')
print("Đã xuất xong 2 file vào thư mục data/processed/")

Đã xuất xong 2 file vào thư mục data/processed/
